# UVA DS4002 Project 1 - Movie Sentiment

In [1]:
import numpy as np
import pandas as pd

### Sentiment Analysis with VADER

NOTE: Be sure to install/upgrade the VADER sentiment package prior to usage. Instructions can be found here: https://vadersentiment.readthedocs.io/en/latest/pages/installation.html

In [ ]:
# Preliminary Note!!!!: If you get any error, first try addressing the code under each "Note" chunk of the code

# --- VADER SENTIMENT ANALYSIS SECTION ---
# This section calculates sentiment scores for each review using the VADER library

# Step 1: Install VADER sentiment library
#VADER PACKAGE ANALYSIS (THIS SECTION IS COMMENTED OUT TO SAVE TIME DUE TO EXCESSIVE RUNTIME)
#!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


df = pd.read_csv('../DATA/imdb_sup.csv')  # Loads dataset containing IMDb reviews
# NOTE: If running without the folder/file structure as described in the README, comment out the above line and uncomment the line below
#df_fin = pd.read_csv('imdb_sup.csv')

analyzer = SentimentIntensityAnalyzer() # Initializes VADER sentiment analyzer object


# Calculates 'compound' score (a normalized, weighted composite score) for each review in the "Review" column
df["compound"] = df["Review"].apply(
    lambda x: analyzer.polarity_scores(x)["compound"]
)


# Defines helper function to convert the continuous compound score into discrete labels
def vader_label(score):
    if score >= 0.05:
        return 1      # positive
    elif score <= -0.05:
        return -1     # negative
    else:
        return 0      # neutral


# Applies labeling function to create a final "Sentiment" column
df["Sentiment"] = df["compound"].apply(vader_label)

# Save the results to a new CSV in the DATA folder to avoid re-running the sentiment analysis later
df.to_csv('../DATA/imdb_reviews_compound.csv', index=False)
# NOTE: If running without the folder/file structure as described in the README, comment out the above line and uncomment the line below
#df_fin = pd.read_csv('imdb_reviews_compound.csv')

### Analysis

NOTE: To install statsmodels follow instructions here: https://www.statsmodels.org/dev/install.html.

To install sklearn follow instructions here: https://scikit-learn.org/stable/install.html.

In [ ]:
# ANALYSIS
df_fin = pd.read_csv('../DATA/imdb_reviews_compound.csv')
# NOTE: If running without the folder/file structure as described in the README, comment out the above line and uncomment the line below
#df_fin = pd.read_csv('imdb_reviews_compound.csv')

from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.metrics import mean_absolute_error, r2_score

In [8]:
# 1. Prepare your variables
# Ensure Rating is sorted and treated as an integer category
df_fin['Rating'] = df_fin['Rating'].astype(int)
X = df_fin[['compound']]
y = df_fin['Rating']

In [9]:
# 2. Fit the Ordinal Regression Model
# 'logit' corresponds to Ordered Logistic Regression
model_ord = OrderedModel(y, X, distr='logit')
res_ord = model_ord.fit(method='bfgs', disp=False)

In [10]:
# 3. Print the Hypothesis Test results
# This includes the coefficients for 'compound' and the "thresholds" between ratings
print(res_ord.summary())

                             OrderedModel Results                             
Dep. Variable:                 Rating   Log-Likelihood:                -95353.
Model:                   OrderedModel   AIC:                         1.907e+05
Method:            Maximum Likelihood   BIC:                         1.908e+05
Date:                Mon, 09 Feb 2026                                         
Time:                        10:17:27                                         
No. Observations:               50000                                         
Df Residuals:                   49992                                         
Df Model:                           1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
compound       1.1295      0.011    104.176      0.000       1.108       1.151
1/2           -1.2720      0.012   -107.606      0.0

In [11]:
# 4. Calculate Evaluation Metrics
# Ordinal models predict probabilities for EACH rating level.
# To get a single "predicted rating" for MAE and R2, we calculate the Expected Value.
pred_probs = res_ord.predict(X)
unique_ratings = np.sort(y.unique())
# Expected Value = (Prob of 1 * 1) + (Prob of 2 * 2) ...
predictions = pred_probs.dot(unique_ratings)

In [12]:
mae = mean_absolute_error(y, predictions)
r2 = r2_score(y, predictions)

print(f"\n--- Evaluation ---")
print(f"Mean Absolute Error (MAE): {mae:.3f}")
print(f"R-squared (R2): {r2:.3f}")



--- Evaluation ---
Mean Absolute Error (MAE): 2.606
R-squared (R2): 0.227


In [ ]:
# NOTE: this was the code used to carry out the original Ordinary Least Squares analysis before we moved on to the ordinal analyis
# To run this code below, just comment it out



# import statsmodels.api as sm

# # 1. Prepare variables for Linear Regression
# # Standard OLS requires an explicit intercept (constant) to be added
# X_linear = sm.add_constant(df_fin['compound'])
# y_linear = df_fin['Rating']

# # 2. Fit the Regular Linear Regression Model
# model_linear = sm.OLS(y_linear, X_linear)
# res_linear = model_linear.fit()

# # 3. Calculate Predictions and Metrics
# linear_predictions = res_linear.predict(X_linear)

# linear_mae = mean_absolute_error(y_linear, linear_predictions)
# linear_r2 = res_linear.rsquared

# print(f"--- Linear Regression Results ---")
# print(res_linear.summary())
# print(f"\nLinear MAE: {linear_mae:.3f}")
# print(f"Linear R2: {linear_r2:.3f}")